March Madness / college basketball prediction project centered on building a model that predicts NCAA tournament games from historical team-level data.

In [1]:
import bisect
from collections import defaultdict
import time
import warnings

import numpy as np
import optuna
import pandas as pd
import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.base import clone
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

warnings.filterwarnings("ignore", message="X does not have valid feature names")

In [2]:
CURRENT_SEASON = 2026
DATA_DIR = 'march-machine-learning-mania-2026'
GENDERS = ['M', 'W']
VALIDATION_SEASONS = list(range(2016, 2026))

# LR needs tournament upweighting because its single linear boundary is dominated by
# regular-season volume; tree models learn the tournament regime natively (weight=1 wins).
LR_TOURNAMENT_WEIGHT = 50

TOURNAMENT_DAY = 136  # approximate first round day
ORDINAL_RANK_FILL = 176  # neutral midpoint for teams with no ranking yet

# Ordinal ranking systems with perfect 24-season coverage (2003-2026)
# Used for computing cross-system stats (std dev, min rank)
ORDINAL_SYSTEMS = ['COL', 'DOL', 'MOR', 'POM', 'WLK']

# Subset exposed as individual features (maximizes methodological diversity)
# POM = adjusted efficiency, MOR = margin + SOS (highest deviation), COL = pure win/loss
ORDINAL_INDIVIDUAL_SYSTEMS = ['POM', 'MOR', 'COL']

# Allowlist of base features to include in training
# Raw counting stats excluded — their signal is captured by rates/efficiencies
ALLOWED_FEATURES = {
    # Shooting rates (pace-independent)
    'fg_pct', 'fg3_pct', 'ft_pct', 'fg3_rate',
    'opponent_fg_pct', 'opponent_fg3_pct', 'opponent_ft_pct', 'opponent_fg3_rate',
    # Dean Oliver's Four Factors
    'offensive_efg_pct', 'defensive_efg_pct',
    'offensive_tov_pct', 'defensive_tov_pct',
    'offensive_reb_pct', 'defensive_reb_pct',
    'offensive_ft_rate', 'defensive_ft_rate',
    # Efficiency ratings (Bayesian-updated, SOS-adjusted)
    'offensive_efficiency', 'defensive_efficiency',
    # Tempo
    'avg_possessions',
    # Defensive activity (own stats — more stable than opponent versions)
    'avg_stl', 'avg_blk',
    # Rebounds (raw — may remove in Tier 2)
    'avg_or', 'avg_dr', 'avg_opponent_or', 'avg_opponent_dr',
    # Turnovers (raw — may remove in Tier 2)
    'avg_to', 'avg_opponent_to',
    # Assists (raw — may remove in Tier 2)
    'avg_ast', 'avg_opponent_ast',
}

In [3]:
data = {os.path.splitext(f)[0]: pd.read_csv(os.path.join(DATA_DIR, f))
        for f in sorted(os.listdir(DATA_DIR)) if f.endswith('.csv')}

In [4]:
print(data[f'{GENDERS[0]}RegularSeasonDetailedResults'].columns)

Index(['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc',
       'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR',
       'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3',
       'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'],
      dtype='str')


In [5]:
BOX_STATS = ['Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
             'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']

PRIOR_OFFENSIVE_EFFICIENCY = 100.0
PRIOR_DEFENSIVE_EFFICIENCY = 100.0
EFFICIENCY_LEARNING_RATE = 0.10
LEAGUE_PRIOR_GAMES = 200
AVERAGE_POSSESSIONS_PER_TEAM = 69.0
HOME_COURT_ADVANTAGE = 3.5  # points per 100 possessions
SEASON_CARRYOVER_WEIGHT = 0.5
RESIDUAL_CAP = 15.0


def estimate_possessions(game, prefix):
    # FGA - OR + TO + 0.475 * FTA (standard Kenpom possession estimate)
    return game[f'{prefix}FGA'] - game[f'{prefix}OR'] + game[f'{prefix}TO'] + 0.475 * game[f'{prefix}FTA']


def update_team_box_stats(entry, game, own_prefix, opp_prefix):
    entry['game_count'] += 1
    for stat in BOX_STATS:
        entry['sums'][stat] += game[f'{own_prefix}{stat}']
        entry['sums'][f'opponent_{stat}'] += game[f'{opp_prefix}{stat}']


def extract_team_features(entry):
    """Extract features from a team's rolling entry."""
    game_count = entry['game_count']
    sums = entry['sums']
    features = {}

    # 28 per-game averages (14 own + 14 opponent)
    for stat in BOX_STATS:
        features[f'avg_{stat.lower()}'] = sums[stat] / game_count
        features[f'avg_opponent_{stat.lower()}'] = sums[f'opponent_{stat}'] / game_count

    # 8 rate stats (from sums for proper weighting)
    def safe_divide(numerator, denominator):
        return numerator / denominator if denominator > 0 else 0.0

    features['fg_pct'] = safe_divide(sums['FGM'], sums['FGA'])
    features['fg3_pct'] = safe_divide(sums['FGM3'], sums['FGA3'])
    features['ft_pct'] = safe_divide(sums['FTM'], sums['FTA'])
    features['fg3_rate'] = safe_divide(sums['FGA3'], sums['FGA'])
    features['opponent_fg_pct'] = safe_divide(sums['opponent_FGM'], sums['opponent_FGA'])
    features['opponent_fg3_pct'] = safe_divide(sums['opponent_FGM3'], sums['opponent_FGA3'])
    features['opponent_ft_pct'] = safe_divide(sums['opponent_FTM'], sums['opponent_FTA'])
    features['opponent_fg3_rate'] = safe_divide(sums['opponent_FGA3'], sums['opponent_FGA'])

    # Dean Oliver's Four Factors (offense and defense)
    # 1. Effective Field Goal % — eFG% = (FG + 0.5 * 3P) / FGA
    features['offensive_efg_pct'] = safe_divide(sums['FGM'] + 0.5 * sums['FGM3'], sums['FGA'])
    features['defensive_efg_pct'] = safe_divide(sums['opponent_FGM'] + 0.5 * sums['opponent_FGM3'], sums['opponent_FGA'])

    # 2. Turnover % — TOV% = TOV / (FGA + 0.44 * FTA + TOV)
    features['offensive_tov_pct'] = safe_divide(sums['TO'], sums['FGA'] + 0.44 * sums['FTA'] + sums['TO'])
    features['defensive_tov_pct'] = safe_divide(sums['opponent_TO'], sums['opponent_FGA'] + 0.44 * sums['opponent_FTA'] + sums['opponent_TO'])

    # 3. Rebound % — ORB% = ORB / (ORB + Opp DRB), DRB% = DRB / (Opp ORB + DRB)
    features['offensive_reb_pct'] = safe_divide(sums['OR'], sums['OR'] + sums['opponent_DR'])
    features['defensive_reb_pct'] = safe_divide(sums['DR'], sums['opponent_OR'] + sums['DR'])

    # 4. Free Throw Rate — FT / FGA
    features['offensive_ft_rate'] = safe_divide(sums['FTM'], sums['FGA'])
    features['defensive_ft_rate'] = safe_divide(sums['opponent_FTM'], sums['opponent_FGA'])

    # 3 efficiency
    features['offensive_efficiency'] = entry['offensive_efficiency']
    features['defensive_efficiency'] = entry['defensive_efficiency']
    features['net_efficiency'] = entry['offensive_efficiency'] - entry['defensive_efficiency']

    # 1 tempo
    features['avg_possessions'] = entry['total_possessions'] / game_count

    return features


def featurize_game(team_a_entry, team_b_entry, season, day_number, team_a_id, team_b_id,
                   label, team_a_location, is_tournament, team_a_seed, team_b_seed,
                   team_a_ordinals, team_b_ordinals,
                   team_a_features=None, team_b_features=None):
    """Build a feature row from two teams' rolling entries (team_a = lower ID).

    Returns None if either team has no games yet.
    Accepts optional pre-computed features to avoid redundant extraction.
    """
    if team_a_entry['game_count'] == 0 or team_b_entry['game_count'] == 0:
        return None

    if team_a_features is None:
        team_a_features = extract_team_features(team_a_entry)
    if team_b_features is None:
        team_b_features = extract_team_features(team_b_entry)

    row = {}
    for key, value in team_a_features.items():
        row[f'team_a_{key}'] = value
    for key, value in team_b_features.items():
        row[f'team_b_{key}'] = value
    for key in team_a_features:
        row[f'diff_{key}'] = team_a_features[key] - team_b_features[key]

    row['team_a_seed'] = team_a_seed
    row['team_b_seed'] = team_b_seed

    for key, value in team_a_ordinals.items():
        row[f'team_a_{key}'] = value
    for key, value in team_b_ordinals.items():
        row[f'team_b_{key}'] = value

    row['season'] = season
    row['day_number'] = day_number
    row['team_a_id'] = team_a_id
    row['team_b_id'] = team_b_id
    row['label'] = label
    row['team_a_location'] = team_a_location
    row['is_tournament'] = is_tournament

    return row


def build_pipeline(gender):
    """Build training data and prediction helper for one gender.

    Returns dict with training_data, feature_columns, feature_columns_no_seed,
    seed_lookup, predict_matchup, precompute_team_data, and has_ordinals.
    """
    has_ordinals = f'{gender}MasseyOrdinals' in data

    # Load gender-specific data
    regular_season_results = data[f'{gender}RegularSeasonDetailedResults'].copy()
    regular_season_results['is_tournament'] = 0
    tournament_results = data[f'{gender}NCAATourneyDetailedResults'].copy()
    tournament_results['is_tournament'] = 1
    results = pd.concat([regular_season_results, tournament_results], ignore_index=True).sort_values(
        ['Season', 'DayNum']).reset_index(drop=True)

    # Build tournament seed lookup: (season, team_id) -> seed number (1-16)
    seed_lookup = {}
    for _, seed_row in data[f'{gender}NCAATourneySeeds'].iterrows():
        seed_number = int(seed_row['Seed'][1:3])  # "W01" -> 1, "X16a" -> 16
        seed_lookup[(seed_row['Season'], seed_row['TeamID'])] = seed_number

    # Build ordinal lookups (only when ordinal data exists)
    if has_ordinals:
        ordinal_averages = (data[f'{gender}MasseyOrdinals']
            .groupby(['Season', 'RankingDayNum', 'TeamID'])['OrdinalRank']
            .mean()
            .reset_index())

        ordinal_lookup = defaultdict(list)
        for _, row in ordinal_averages.iterrows():
            ordinal_lookup[(row['Season'], row['TeamID'])].append(
                (row['RankingDayNum'], row['OrdinalRank']))
        for key in ordinal_lookup:
            ordinal_lookup[key].sort()

        selected_ordinals = data[f'{gender}MasseyOrdinals'][
            data[f'{gender}MasseyOrdinals']['SystemName'].isin(ORDINAL_SYSTEMS)]

        system_ordinal_lookup = defaultdict(list)
        for _, row in selected_ordinals.iterrows():
            system_ordinal_lookup[(row['SystemName'], row['Season'], row['TeamID'])].append(
                (row['RankingDayNum'], row['OrdinalRank']))
        for key in system_ordinal_lookup:
            system_ordinal_lookup[key].sort()

        ordinal_stats_grouped = (selected_ordinals
            .groupby(['Season', 'RankingDayNum', 'TeamID'])['OrdinalRank']
            .agg(['std', 'min'])
            .reset_index())

        ordinal_stats_lookup = defaultdict(list)
        for _, row in ordinal_stats_grouped.iterrows():
            std_value = row['std'] if not np.isnan(row['std']) else 0.0
            ordinal_stats_lookup[(row['Season'], row['TeamID'])].append(
                (row['RankingDayNum'], std_value, row['min']))
        for key in ordinal_stats_lookup:
            ordinal_stats_lookup[key].sort()

        print(f"[{gender}] Ordinal lookup: {len(ordinal_lookup)}, system: {len(system_ordinal_lookup)}, stats: {len(ordinal_stats_lookup)}")
    else:
        ordinal_lookup = {}
        system_ordinal_lookup = {}
        ordinal_stats_lookup = {}
        print(f"[{gender}] No ordinal data available")

    # --- Ordinal helper closures ---

    def get_ordinal_rank(season, team_id, day_num):
        rankings = ordinal_lookup.get((season, team_id))
        if not rankings:
            return np.nan
        day_nums = [r[0] for r in rankings]
        idx = bisect.bisect_right(day_nums, day_num) - 1
        if idx < 0:
            return np.nan
        return rankings[idx][1]

    def get_system_ordinal_rank(system_name, season, team_id, day_num):
        rankings = system_ordinal_lookup.get((system_name, season, team_id))
        if not rankings:
            return np.nan
        day_nums = [r[0] for r in rankings]
        idx = bisect.bisect_right(day_nums, day_num) - 1
        if idx < 0:
            return np.nan
        return rankings[idx][1]

    def get_ordinal_stats(season, team_id, day_num):
        stats = ordinal_stats_lookup.get((season, team_id))
        if not stats:
            return np.nan, np.nan
        day_nums = [r[0] for r in stats]
        idx = bisect.bisect_right(day_nums, day_num) - 1
        if idx < 0:
            return np.nan, np.nan
        return stats[idx][1], stats[idx][2]

    def build_ordinal_features(season, team_id, day_num):
        if not has_ordinals:
            return {}
        features = {}
        features['ordinal_rank'] = get_ordinal_rank(season, team_id, day_num)
        for system_name in ORDINAL_INDIVIDUAL_SYSTEMS:
            features[f'{system_name.lower()}_rank'] = get_system_ordinal_rank(system_name, season, team_id, day_num)
        std_dev, min_rank = get_ordinal_stats(season, team_id, day_num)
        features['ordinal_std'] = std_dev
        features['ordinal_min'] = min_rank
        return features

    # --- Rolling stats closures ---

    team_rolling = {}
    league_totals = {}

    def get_or_create_entry(season, team_id):
        key = (season, team_id)
        if key not in team_rolling:
            sums = {}
            for stat in BOX_STATS:
                sums[stat] = 0.0
                sums[f'opponent_{stat}'] = 0.0

            prior_key = (season - 1, team_id)
            if prior_key in team_rolling:
                prior = team_rolling[prior_key]
                initial_offensive = (SEASON_CARRYOVER_WEIGHT * prior['offensive_efficiency']
                                     + (1 - SEASON_CARRYOVER_WEIGHT) * PRIOR_OFFENSIVE_EFFICIENCY)
                initial_defensive = (SEASON_CARRYOVER_WEIGHT * prior['defensive_efficiency']
                                     + (1 - SEASON_CARRYOVER_WEIGHT) * PRIOR_DEFENSIVE_EFFICIENCY)
            else:
                initial_offensive = PRIOR_OFFENSIVE_EFFICIENCY
                initial_defensive = PRIOR_DEFENSIVE_EFFICIENCY

            team_rolling[key] = {
                'sums': sums,
                'game_count': 0,
                'offensive_efficiency': initial_offensive,
                'defensive_efficiency': initial_defensive,
                'total_possessions': 0.0,
            }
        return team_rolling[key]

    def get_or_create_league_totals(season):
        if season not in league_totals:
            league_totals[season] = {'total_points': 0.0, 'total_possessions': 0.0, 'game_count': 0}
        return league_totals[season]

    def get_league_average_efficiency(season):
        totals = get_or_create_league_totals(season)
        phantom_games = max(0, LEAGUE_PRIOR_GAMES - totals['game_count'])
        if phantom_games == 0:
            return totals['total_points'] / totals['total_possessions'] * 100.0
        if season - 1 in league_totals and league_totals[season - 1]['total_possessions'] > 0:
            prior_efficiency = (league_totals[season - 1]['total_points']
                                / league_totals[season - 1]['total_possessions'] * 100.0)
        else:
            prior_efficiency = 100.0
        phantom_possessions = phantom_games * 2 * AVERAGE_POSSESSIONS_PER_TEAM
        phantom_points = phantom_possessions * prior_efficiency / 100.0
        blended_points = phantom_points + totals['total_points']
        blended_possessions = phantom_possessions + totals['total_possessions']
        return blended_points / blended_possessions * 100.0

    # --- Main game-processing loop ---

    training_rows = []

    for _, game in results.iterrows():
        winner_entry = get_or_create_entry(game['Season'], game['WTeamID'])
        loser_entry = get_or_create_entry(game['Season'], game['LTeamID'])

        winner_team_id = game['WTeamID']
        loser_team_id = game['LTeamID']

        if game['is_tournament'] == 1:
            winner_seed = seed_lookup.get((game['Season'], winner_team_id), np.nan)
            loser_seed = seed_lookup.get((game['Season'], loser_team_id), np.nan)
        else:
            winner_seed = np.nan
            loser_seed = np.nan

        winner_ordinals = build_ordinal_features(game['Season'], winner_team_id, game['DayNum'])
        loser_ordinals = build_ordinal_features(game['Season'], loser_team_id, game['DayNum'])

        if winner_team_id < loser_team_id:
            team_a_location = game['WLoc']
            row = featurize_game(winner_entry, loser_entry, game['Season'], game['DayNum'],
                                 winner_team_id, loser_team_id, label=1, team_a_location=team_a_location,
                                 is_tournament=game['is_tournament'],
                                 team_a_seed=winner_seed, team_b_seed=loser_seed,
                                 team_a_ordinals=winner_ordinals, team_b_ordinals=loser_ordinals)
        else:
            location_flip = {'H': 'A', 'A': 'H', 'N': 'N'}
            team_a_location = location_flip[game['WLoc']]
            row = featurize_game(loser_entry, winner_entry, game['Season'], game['DayNum'],
                                 loser_team_id, winner_team_id, label=0, team_a_location=team_a_location,
                                 is_tournament=game['is_tournament'],
                                 team_a_seed=loser_seed, team_b_seed=winner_seed,
                                 team_a_ordinals=loser_ordinals, team_b_ordinals=winner_ordinals)
        if row is not None:
            training_rows.append(row)

        # --- Adjusted efficiency update ---
        possessions = (estimate_possessions(game, 'W') + estimate_possessions(game, 'L')) / 2

        if possessions > 0:
            winner_raw_offensive = game['WScore'] / possessions * 100.0
            loser_raw_offensive = game['LScore'] / possessions * 100.0

            if game['WLoc'] == 'H':
                winner_raw_offensive -= HOME_COURT_ADVANTAGE
                loser_raw_offensive += HOME_COURT_ADVANTAGE
            elif game['WLoc'] == 'A':
                winner_raw_offensive += HOME_COURT_ADVANTAGE
                loser_raw_offensive -= HOME_COURT_ADVANTAGE

            league_average = get_league_average_efficiency(game['Season'])

            winner_expected_offensive = (winner_entry['offensive_efficiency']
                                         + loser_entry['defensive_efficiency']
                                         - league_average)
            loser_expected_offensive = (loser_entry['offensive_efficiency']
                                        + winner_entry['defensive_efficiency']
                                        - league_average)

            winner_offensive_residual = np.clip(winner_raw_offensive - winner_expected_offensive, -RESIDUAL_CAP, RESIDUAL_CAP)
            loser_offensive_residual = np.clip(loser_raw_offensive - loser_expected_offensive, -RESIDUAL_CAP, RESIDUAL_CAP)

            winner_entry['offensive_efficiency'] += EFFICIENCY_LEARNING_RATE * winner_offensive_residual
            loser_entry['defensive_efficiency'] += EFFICIENCY_LEARNING_RATE * winner_offensive_residual
            loser_entry['offensive_efficiency'] += EFFICIENCY_LEARNING_RATE * loser_offensive_residual
            winner_entry['defensive_efficiency'] += EFFICIENCY_LEARNING_RATE * loser_offensive_residual

        season_totals = get_or_create_league_totals(game['Season'])
        season_totals['total_points'] += game['WScore'] + game['LScore']
        season_totals['total_possessions'] += 2 * possessions
        season_totals['game_count'] += 1

        winner_entry['total_possessions'] += possessions
        loser_entry['total_possessions'] += possessions

        update_team_box_stats(winner_entry, game, 'W', 'L')
        update_team_box_stats(loser_entry, game, 'L', 'W')

    print(f"[{gender}] Processed {len(results)} games, {len(team_rolling)} team-seasons")
    print(f"[{gender}] Training rows: {len(training_rows)}, seed lookup: {len(seed_lookup)}")

    # --- Build training DataFrame ---

    training_data = pd.DataFrame(training_rows)

    # --- Feature selection ---

    ordinal_fill_columns = {'rank': [], 'std': [], 'min': []}

    if has_ordinals:
        ordinal_rank_columns = [col for col in training_data.columns
                                if col.endswith('_rank') and ('ordinal' in col or 'pom' in col or 'mor' in col or 'col' in col)]
        ordinal_std_columns = [col for col in training_data.columns if col.endswith('_ordinal_std')]
        ordinal_min_columns = [col for col in training_data.columns if col.endswith('_ordinal_min')]

        for col in ordinal_rank_columns:
            training_data[col] = training_data[col].fillna(ORDINAL_RANK_FILL)
        for col in ordinal_std_columns:
            training_data[col] = training_data[col].fillna(0.0)
        for col in ordinal_min_columns:
            training_data[col] = training_data[col].fillna(ORDINAL_RANK_FILL)

        ordinal_fill_columns = {'rank': ordinal_rank_columns, 'std': ordinal_std_columns, 'min': ordinal_min_columns}

        print(f"[{gender}] Ordinal rank coverage: {(training_data['team_a_ordinal_rank'] != ORDINAL_RANK_FILL).mean():.1%}")

    allowed_columns = set()
    for base_name in ALLOWED_FEATURES:
        allowed_columns.add(f'team_a_{base_name}')
        allowed_columns.add(f'team_b_{base_name}')
    if has_ordinals:
        ordinal_feature_names = {'ordinal_rank', 'ordinal_std', 'ordinal_min'}
        for sys in ORDINAL_INDIVIDUAL_SYSTEMS:
            ordinal_feature_names.add(f'{sys.lower()}_rank')
        for base_name in ordinal_feature_names:
            allowed_columns.add(f'team_a_{base_name}')
            allowed_columns.add(f'team_b_{base_name}')
    allowed_columns.add('team_a_seed')
    allowed_columns.add('team_b_seed')

    metadata_columns = {"season", "day_number", "team_a_id", "team_b_id", "label",
                        "team_a_location", "is_tournament"}
    feature_columns = [col for col in training_data.columns
                       if col in allowed_columns and col not in metadata_columns]

    seed_columns = {"team_a_seed", "team_b_seed"}
    feature_columns_no_seed = [col for col in feature_columns if col not in seed_columns]

    print(f"[{gender}] {len(feature_columns)} features (incl seed), {len(feature_columns_no_seed)} without seed")

    # --- Pre-computation helper ---

    def precompute_team_data(team_ids):
        """Pre-compute features and ordinals for a set of teams."""
        features_cache = {}
        ordinals_cache = {}
        for team_id in team_ids:
            key = (CURRENT_SEASON, team_id)
            if key not in team_rolling:
                continue
            features_cache[team_id] = extract_team_features(team_rolling[key])
            ordinals_cache[team_id] = build_ordinal_features(CURRENT_SEASON, team_id, TOURNAMENT_DAY)
        return features_cache, ordinals_cache

    # --- Prediction closure ---

    def predict_matchup(model, team_id_1, team_id_2, seed_num_1, seed_num_2,
                        precomputed_features=None, precomputed_ordinals=None):
        """Predict win probability for team_id_1 using bidirectional averaging."""
        entry_1 = team_rolling[(CURRENT_SEASON, team_id_1)]
        entry_2 = team_rolling[(CURRENT_SEASON, team_id_2)]

        if precomputed_features is not None:
            features_1 = precomputed_features[team_id_1]
            features_2 = precomputed_features[team_id_2]
        else:
            features_1 = extract_team_features(entry_1)
            features_2 = extract_team_features(entry_2)

        if precomputed_ordinals is not None:
            ordinals_1 = precomputed_ordinals[team_id_1]
            ordinals_2 = precomputed_ordinals[team_id_2]
        else:
            ordinals_1 = build_ordinal_features(CURRENT_SEASON, team_id_1, TOURNAMENT_DAY)
            ordinals_2 = build_ordinal_features(CURRENT_SEASON, team_id_2, TOURNAMENT_DAY)

        lower_id = min(team_id_1, team_id_2)
        higher_id = max(team_id_1, team_id_2)

        if team_id_1 < team_id_2:
            lower_entry, higher_entry = entry_1, entry_2
            lower_seed, higher_seed = seed_num_1, seed_num_2
            lower_ordinals, higher_ordinals = ordinals_1, ordinals_2
            lower_features, higher_features = features_1, features_2
        else:
            lower_entry, higher_entry = entry_2, entry_1
            lower_seed, higher_seed = seed_num_2, seed_num_1
            lower_ordinals, higher_ordinals = ordinals_2, ordinals_1
            lower_features, higher_features = features_2, features_1

        row_forward = featurize_game(
            lower_entry, higher_entry, CURRENT_SEASON, TOURNAMENT_DAY,
            lower_id, higher_id, label=0, team_a_location='N',
            is_tournament=1, team_a_seed=lower_seed, team_b_seed=higher_seed,
            team_a_ordinals=lower_ordinals, team_b_ordinals=higher_ordinals,
            team_a_features=lower_features, team_b_features=higher_features)

        row_reversed = featurize_game(
            higher_entry, lower_entry, CURRENT_SEASON, TOURNAMENT_DAY,
            higher_id, lower_id, label=0, team_a_location='N',
            is_tournament=1, team_a_seed=higher_seed, team_b_seed=lower_seed,
            team_a_ordinals=higher_ordinals, team_b_ordinals=lower_ordinals,
            team_a_features=higher_features, team_b_features=lower_features)

        prediction_rows = pd.DataFrame([row_forward, row_reversed])
        for col in ordinal_fill_columns['rank']:
            prediction_rows[col] = prediction_rows[col].fillna(ORDINAL_RANK_FILL)
        for col in ordinal_fill_columns['std']:
            prediction_rows[col] = prediction_rows[col].fillna(0.0)
        for col in ordinal_fill_columns['min']:
            prediction_rows[col] = prediction_rows[col].fillna(ORDINAL_RANK_FILL)

        X_forward = prediction_rows.iloc[[0]][feature_columns].values
        X_reversed = prediction_rows.iloc[[1]][feature_columns].values

        prob_lower_wins_forward = model.predict_proba(X_forward)[0, 1]
        prob_higher_wins_reversed = model.predict_proba(X_reversed)[0, 1]

        prob_lower_wins = (prob_lower_wins_forward + (1 - prob_higher_wins_reversed)) / 2

        if team_id_1 == lower_id:
            return prob_lower_wins
        else:
            return 1 - prob_lower_wins

    return {
        'training_data': training_data,
        'feature_columns': feature_columns,
        'feature_columns_no_seed': feature_columns_no_seed,
        'seed_lookup': seed_lookup,
        'predict_matchup': predict_matchup,
        'precompute_team_data': precompute_team_data,
        'has_ordinals': has_ordinals,
    }


pipelines = {gender: build_pipeline(gender) for gender in GENDERS}

[M] Ordinal lookup: 8356, system: 41729, stats: 8355
[M] Processed 125978 games, 8346 team-seasons
[M] Training rows: 120861, seed lookup: 2694
[M] Ordinal rank coverage: 97.3%
[M] 72 features (incl seed), 70 without seed
[W] No ordinal data available
[W] Processed 88148 games, 5965 team-seasons
[W] Training rows: 84535, seed lookup: 1812
[W] 60 features (incl seed), 58 without seed


### Cross-validation results (model selection)

LightGBM was selected based on these results. The code is commented out below.

```
=== M: Cross-validation (10 folds, seasons 2016–2025) ===

Model                  Feats            ROC AUC              Brier        Tourney AUC      Tourney Brier
======================================================================================================
LogisticRegression        70 0.7791 ± 0.0083   0.1920 ± 0.0038   0.7698 ± 0.0553   0.1925 ± 0.0204
HistGradientBoosting      72 0.7806 ± 0.0083   0.1903 ± 0.0035   0.7775 ± 0.0541   0.1911 ± 0.0204
XGBoost                   72 0.7806 ± 0.0088   0.1903 ± 0.0037   0.7742 ± 0.0527   0.1913 ± 0.0205
LightGBM                  72 0.7803 ± 0.0083   0.1905 ± 0.0035   0.7730 ± 0.0537   0.1908 ± 0.0218

=== W: Cross-validation (10 folds, seasons 2016–2025) ===

Model                  Feats            ROC AUC              Brier        Tourney AUC      Tourney Brier
======================================================================================================
LogisticRegression        58 0.8284 ± 0.0100   0.1712 ± 0.0089   0.8636 ± 0.0343   0.1606 ± 0.0268
HistGradientBoosting      60 0.8309 ± 0.0089   0.1678 ± 0.0044   0.8662 ± 0.0386   0.1552 ± 0.0241
XGBoost                   60 0.8308 ± 0.0090   0.1679 ± 0.0045   0.8684 ± 0.0361   0.1543 ± 0.0241
LightGBM                  60 0.8306 ± 0.0091   0.1680 ± 0.0045   0.8627 ± 0.0335   0.1576 ± 0.0243
```

In [6]:
# # --- Cross-validation ---
#
# VALIDATION_SEASONS = list(range(2016, 2026))
#
# models = {
#     "LogisticRegression": LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs"),
#     "HistGradientBoosting": HistGradientBoostingClassifier(
#         max_iter=200, max_depth=4, learning_rate=0.1, random_state=42,
#     ),
#     "XGBoost": XGBClassifier(
#         n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42,
#         eval_metric="logloss",
#     ),
#     "LightGBM": LGBMClassifier(
#         n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42,
#         verbose=-1,
#     ),
# }
#
# for gender in GENDERS:
#     pipeline = pipelines[gender]
#     training_data = pipeline['training_data']
#     feature_columns = pipeline['feature_columns']
#     feature_columns_no_seed = pipeline['feature_columns_no_seed']
#
#     y = training_data["label"].values
#     seasons = training_data["season"].values
#     is_tournament = training_data["is_tournament"].values
#     lr_sample_weights = np.where(is_tournament == 1, LR_TOURNAMENT_WEIGHT, 1.0)
#
#     folds = []
#     for holdout_season in VALIDATION_SEASONS:
#         train_mask = seasons < holdout_season
#         validation_mask = seasons == holdout_season
#         if train_mask.sum() > 0 and validation_mask.sum() > 0:
#             folds.append((np.where(train_mask)[0], np.where(validation_mask)[0]))
#
#     results_table = []
#
#     for model_name, model_template in models.items():
#         fold_aucs, fold_briers = [], []
#         tournament_aucs, tournament_briers = [], []
#
#         if model_name == "LogisticRegression":
#             model_features = feature_columns_no_seed
#             use_scaler = True
#         else:
#             model_features = feature_columns
#             use_scaler = False
#
#         X_model = training_data[model_features].values
#
#         for fold_index, (train_indices, validation_indices) in enumerate(folds):
#             if use_scaler:
#                 scaler = StandardScaler()
#                 X_train = scaler.fit_transform(X_model[train_indices])
#                 X_validation = scaler.transform(X_model[validation_indices])
#             else:
#                 X_train = X_model[train_indices]
#                 X_validation = X_model[validation_indices]
#
#             y_train = y[train_indices]
#             y_validation = y[validation_indices]
#
#             model = clone(model_template)
#             fit_kwargs = {}
#             if model_name == "LogisticRegression":
#                 fit_kwargs["sample_weight"] = lr_sample_weights[train_indices]
#             model.fit(X_train, y_train, **fit_kwargs)
#             probabilities = model.predict_proba(X_validation)[:, 1]
#
#             fold_aucs.append(roc_auc_score(y_validation, probabilities))
#             fold_briers.append(brier_score_loss(y_validation, probabilities))
#
#             tournament_mask = is_tournament[validation_indices] == 1
#             if tournament_mask.sum() > 0:
#                 tournament_aucs.append(roc_auc_score(y_validation[tournament_mask], probabilities[tournament_mask]))
#                 tournament_briers.append(brier_score_loss(y_validation[tournament_mask], probabilities[tournament_mask]))
#
#         results_table.append({
#             "model": model_name,
#             "features": len(model_features),
#             "auc_mean": np.mean(fold_aucs),
#             "auc_std": np.std(fold_aucs),
#             "brier_mean": np.mean(fold_briers),
#             "brier_std": np.std(fold_briers),
#             "tournament_auc_mean": np.mean(tournament_aucs),
#             "tournament_auc_std": np.std(tournament_aucs),
#             "tournament_brier_mean": np.mean(tournament_briers),
#             "tournament_brier_std": np.std(tournament_briers),
#         })
#
#     print(f"\n=== {gender}: Cross-validation ({len(folds)} folds, seasons {VALIDATION_SEASONS[0]}–{VALIDATION_SEASONS[-1]}) ===\n")
#     print(f"{'Model':<22} {'Feats':>5} {'ROC AUC':>18} {'Brier':>18} {'Tourney AUC':>18} {'Tourney Brier':>18}")
#     print("=" * 102)
#     for row in results_table:
#         print(f"{row['model']:<22} "
#               f"{row['features']:>5} "
#               f"{row['auc_mean']:.4f} ± {row['auc_std']:.4f}   "
#               f"{row['brier_mean']:.4f} ± {row['brier_std']:.4f}   "
#               f"{row['tournament_auc_mean']:.4f} ± {row['tournament_auc_std']:.4f}   "
#               f"{row['tournament_brier_mean']:.4f} ± {row['tournament_brier_std']:.4f}")

### Optuna Hyperparameter Tuning Results (200 trials)

**M** — 7359.7s (36.8s/trial), 7/200 pruned, best tournament Brier: 0.1883
| Parameter | Value |
|---|---|
| max_depth | 3 |
| learning_rate | 0.0593 |
| num_leaves | 19 |
| min_child_samples | 97 |
| subsample | 0.7835 |
| colsample_bytree | 0.8223 |
| reg_alpha | 3.96e-07 |
| reg_lambda | 2.72e-05 |

**W** — 3795.2s (19.0s/trial), 126/200 pruned, best tournament Brier: 0.1492
| Parameter | Value |
|---|---|
| max_depth | 8 |
| learning_rate | 0.0319 |
| num_leaves | 107 |
| min_child_samples | 19 |
| subsample | 0.8000 |
| colsample_bytree | 0.6700 |
| reg_alpha | 0.2024 |
| reg_lambda | 7.33e-07 |

In [7]:
# # --- Optuna hyperparameter tuning for LightGBM ---
# NUMBER_OF_TUNING_TRIALS = 200
# EARLY_STOPPING_ROUNDS = 50
# N_ESTIMATORS_CEILING = 2000
#
# optuna.logging.set_verbosity(optuna.logging.WARNING)
#
# best_params = {}
#
# for gender in GENDERS:
#     pipeline = pipelines[gender]
#     training_data = pipeline['training_data']
#     feature_columns = pipeline['feature_columns']
#
#     y = training_data["label"].values
#     seasons = training_data["season"].values
#     is_tournament = training_data["is_tournament"].values
#
#     X_tuning = training_data[feature_columns].values
#
#     folds = []
#     for holdout_season in VALIDATION_SEASONS:
#         train_mask = seasons < holdout_season
#         validation_mask = seasons == holdout_season
#         if train_mask.sum() > 0 and validation_mask.sum() > 0:
#             folds.append((np.where(train_mask)[0], np.where(validation_mask)[0]))
#
#     def objective(trial):
#         params = {
#             "n_estimators": N_ESTIMATORS_CEILING,
#             "max_depth": trial.suggest_int("max_depth", 3, 8),
#             "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
#             "num_leaves": trial.suggest_int("num_leaves", 15, 127),
#             "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
#             "subsample": trial.suggest_float("subsample", 0.6, 1.0),
#             "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
#             "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
#             "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
#             "random_state": 42,
#             "verbose": -1,
#         }
#
#         tournament_briers = []
#         for fold_index, (train_indices, validation_indices) in enumerate(folds):
#             model = LGBMClassifier(**params)
#             model.fit(
#                 X_tuning[train_indices], y[train_indices],
#                 eval_set=[(X_tuning[validation_indices], y[validation_indices])],
#                 callbacks=[
#                     early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
#                     log_evaluation(period=0),
#                 ],
#             )
#             probabilities = model.predict_proba(X_tuning[validation_indices])[:, 1]
#
#             tournament_mask = is_tournament[validation_indices] == 1
#             if tournament_mask.sum() > 0:
#                 tournament_briers.append(
#                     brier_score_loss(y[validation_indices][tournament_mask], probabilities[tournament_mask])
#                 )
#
#             if len(tournament_briers) > 0:
#                 trial.report(np.mean(tournament_briers), fold_index)
#                 if trial.should_prune():
#                     raise optuna.TrialPruned()
#
#         return np.mean(tournament_briers)
#
#     study = optuna.create_study(
#         direction="minimize",
#         sampler=optuna.samplers.TPESampler(seed=42),
#         pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=3),
#     )
#
#     start_time = time.time()
#     study.optimize(objective, n_trials=NUMBER_OF_TUNING_TRIALS)
#     elapsed_time = time.time() - start_time
#
#     gender_best = study.best_params
#     gender_best["random_state"] = 42
#     gender_best["verbose"] = -1
#     best_params[gender] = gender_best
#
#     pruned_count = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
#
#     print(f"\n=== {gender} ===")
#     print(f"Tuning: {NUMBER_OF_TUNING_TRIALS} trials in {elapsed_time:.1f}s ({elapsed_time / NUMBER_OF_TUNING_TRIALS:.1f}s/trial)")
#     print(f"  Pruned: {pruned_count}/{NUMBER_OF_TUNING_TRIALS}")
#     print(f"Best tournament Brier: {study.best_value:.4f}")
#     print(f"Best parameters:")
#     for key, value in gender_best.items():
#         print(f"  {key}: {value}")

In [8]:
# --- Train final models (LightGBM on all data with tuned parameters) ---
EARLY_STOPPING_ROUNDS = 50
N_ESTIMATORS_CEILING = 2000

best_params = {
    'M': {
        'max_depth': 3,
        'learning_rate': 0.059333060299498086,
        'num_leaves': 19,
        'min_child_samples': 97,
        'subsample': 0.783493797927359,
        'colsample_bytree': 0.8223133520122138,
        'reg_alpha': 3.9648953931733416e-07,
        'reg_lambda': 2.719806955343662e-05,
        'random_state': 42,
        'verbose': -1,
    },
    'W': {
        'max_depth': 8,
        'learning_rate': 0.031876958529688214,
        'num_leaves': 107,
        'min_child_samples': 19,
        'subsample': 0.7999991686968412,
        'colsample_bytree': 0.6700334587813502,
        'reg_alpha': 0.20238804598217985,
        'reg_lambda': 7.332600615119883e-07,
        'random_state': 42,
        'verbose': -1,
    },
}

final_models = {}

for gender in GENDERS:
    pipeline = pipelines[gender]
    training_data = pipeline['training_data']
    feature_columns = pipeline['feature_columns']

    X_all = training_data[feature_columns].values
    y_all = training_data['label'].values
    seasons_all = training_data['season'].values

    final_train_mask = seasons_all < CURRENT_SEASON - 1
    final_eval_mask = seasons_all == CURRENT_SEASON - 1

    final_params = best_params[gender].copy()
    final_params["n_estimators"] = N_ESTIMATORS_CEILING

    model = LGBMClassifier(**final_params)
    model.fit(
        X_all[final_train_mask], y_all[final_train_mask],
        eval_set=[(X_all[final_eval_mask], y_all[final_eval_mask])],
        callbacks=[
            early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
            log_evaluation(period=0),
        ],
    )
    optimal_trees = model.best_iteration_

    model = LGBMClassifier(**{**final_params, "n_estimators": optimal_trees})
    model.fit(X_all, y_all)
    final_models[gender] = model

    print(f"[{gender}] LightGBM: {optimal_trees} trees (early stopped), trained on {len(y_all)} games")

[M] LightGBM: 608 trees (early stopped), trained on 120861 games
[W] LightGBM: 320 trees (early stopped), trained on 84535 games


In [9]:
# --- 2026 First Round Predictions (Sanity Check) ---

region_labels = {'W': 'WEST', 'X': 'EAST', 'Y': 'SOUTH', 'Z': 'MIDWEST'}
gender_labels = {'M': "MEN'S", 'W': "WOMEN'S"}

for gender in GENDERS:
    pipeline = pipelines[gender]
    seed_lookup = pipeline['seed_lookup']
    predict_matchup = pipeline['predict_matchup']
    model = final_models[gender]

    # Build lookups for current season
    seed_to_team = {}
    for _, row in data[f'{gender}NCAATourneySeeds'][data[f'{gender}NCAATourneySeeds']['Season'] == CURRENT_SEASON].iterrows():
        seed_to_team[row['Seed']] = row['TeamID']

    team_names = dict(zip(data[f'{gender}Teams']['TeamID'], data[f'{gender}Teams']['TeamName']))

    # Resolve play-in games
    slots_current = data[f'{gender}NCAATourneySlots'][data[f'{gender}NCAATourneySlots']['Season'] == CURRENT_SEASON]
    playin_slots = slots_current[~slots_current['Slot'].str.startswith('R')]
    first_round_slots = slots_current[slots_current['Slot'].str.startswith('R1')]

    print(f"\n{'=' * 60}")
    print(f"  {gender_labels[gender]} TOURNAMENT")
    print(f"{'=' * 60}")

    if len(playin_slots) > 0:
        print("\n=== PLAY-IN GAMES ===\n")
        playin_winners = {}
        for _, slot in playin_slots.iterrows():
            strong_team = seed_to_team[slot['StrongSeed']]
            weak_team = seed_to_team[slot['WeakSeed']]
            strong_seed_num = seed_lookup[(CURRENT_SEASON, strong_team)]
            weak_seed_num = seed_lookup[(CURRENT_SEASON, weak_team)]

            prob_strong_wins = predict_matchup(model, strong_team, weak_team, strong_seed_num, weak_seed_num)
            winner_id = strong_team if prob_strong_wins > 0.5 else weak_team
            playin_winners[slot['Slot']] = winner_id

            marker = "←" if prob_strong_wins > 0.5 else " "
            print(f"  {slot['Slot']:4s}: ({strong_seed_num:>2d}) {team_names[strong_team]:<22s} {prob_strong_wins:5.1%}  {marker}")
            marker = " " if prob_strong_wins > 0.5 else "←"
            print(f"        ({weak_seed_num:>2d}) {team_names[weak_team]:<22s} {1 - prob_strong_wins:5.1%}  {marker}")
            print()
    else:
        playin_winners = {}

    # Predict all first round games
    predictions = []

    for _, slot in first_round_slots.iterrows():
        region = slot['Slot'][2]  # R1W1 -> 'W'

        strong_seed_str = slot['StrongSeed']
        strong_team = playin_winners.get(strong_seed_str, seed_to_team.get(strong_seed_str))

        weak_seed_str = slot['WeakSeed']
        weak_team = playin_winners.get(weak_seed_str, seed_to_team.get(weak_seed_str))

        strong_seed_num = seed_lookup[(CURRENT_SEASON, strong_team)]
        weak_seed_num = seed_lookup[(CURRENT_SEASON, weak_team)]

        prob_strong_wins = predict_matchup(model, strong_team, weak_team, strong_seed_num, weak_seed_num)

        predictions.append({
            'region': region,
            'strong_seed': strong_seed_num,
            'weak_seed': weak_seed_num,
            'strong_team': team_names[strong_team],
            'weak_team': team_names[weak_team],
            'strong_seed_win_prob': prob_strong_wins,
        })

    print("\n=== FIRST ROUND PREDICTIONS ===\n")
    for region_code in ['W', 'X', 'Y', 'Z']:
        region_preds = [p for p in predictions if p['region'] == region_code]
        if not region_preds:
            continue
        region_preds.sort(key=lambda p: p['strong_seed'])
        print(f"--- {region_labels[region_code]} ---")
        print(f"  {'Matchup':<52s} {'Fav %':>5s}")
        print(f"  {'─' * 58}")
        for p in region_preds:
            matchup = f"({p['strong_seed']:>2d}) {p['strong_team']:<22s} vs ({p['weak_seed']:>2d}) {p['weak_team']}"
            print(f"  {matchup:<52s} {p['strong_seed_win_prob']:5.1%}")
        print()


  MEN'S TOURNAMENT

=== PLAY-IN GAMES ===

  X16 : (16) Lehigh                 50.6%  ←
        (16) Prairie View           49.4%   

  Y11 : (11) Miami OH               39.3%   
        (11) SMU                    60.7%  ←

  Y16 : (16) Howard                 42.6%   
        (16) UMBC                   57.4%  ←

  Z11 : (11) NC State               47.3%   
        (11) Texas                  52.7%  ←


=== FIRST ROUND PREDICTIONS ===

--- WEST ---
  Matchup                                              Fav %
  ──────────────────────────────────────────────────────────
  ( 1) Duke                   vs (16) Siena            97.5%
  ( 2) Connecticut            vs (15) Furman           93.9%
  ( 3) Michigan St            vs (14) N Dakota St      92.6%
  ( 4) Kansas                 vs (13) Cal Baptist      81.5%
  ( 5) St John's              vs (12) Northern Iowa    81.4%
  ( 6) Louisville             vs (11) South Florida    60.4%
  ( 7) UCLA                   vs (10) UCF              58

In [10]:
# --- Generate submission ---

sample_submission = pd.read_csv(os.path.join(DATA_DIR, 'SampleSubmissionStage2.csv'))
mens_team_ids = set(data['MTeams']['TeamID'])
womens_team_ids = set(data['WTeams']['TeamID'])

# Pre-compute per-team features once per gender
precomputed = {}
for gender in GENDERS:
    pipeline = pipelines[gender]
    team_ids_set = mens_team_ids if gender == 'M' else womens_team_ids
    precomputed[gender] = pipeline['precompute_team_data'](team_ids_set)

predictions = []

for _, row in sample_submission.iterrows():
    parts = row['ID'].split('_')
    team_a_id = int(parts[1])
    team_b_id = int(parts[2])

    if team_a_id in mens_team_ids and team_b_id in mens_team_ids:
        gender = 'M'
    elif team_a_id in womens_team_ids and team_b_id in womens_team_ids:
        gender = 'W'
    else:
        print(f"ERROR: Could not determine gender for {row['ID']}")
        predictions.append(0.5)
        continue

    pipeline = pipelines[gender]
    model = final_models[gender]
    features_cache, ordinals_cache = precomputed[gender]

    seed_a = pipeline['seed_lookup'].get((CURRENT_SEASON, team_a_id), np.nan)
    seed_b = pipeline['seed_lookup'].get((CURRENT_SEASON, team_b_id), np.nan)
    prob = pipeline['predict_matchup'](
        model, team_a_id, team_b_id, seed_a, seed_b,
        precomputed_features=features_cache, precomputed_ordinals=ordinals_cache)
    predictions.append(prob)

sample_submission['Pred'] = predictions
sample_submission[['ID', 'Pred']].to_csv('submission.csv', index=False)

print(f"Submission saved: {len(sample_submission)} rows")
print(f"  Min:  {sample_submission['Pred'].min():.4f}")
print(f"  Max:  {sample_submission['Pred'].max():.4f}")
print(f"  Mean: {sample_submission['Pred'].mean():.4f}")
print(f"\nSample predictions:")
print(sample_submission[['ID', 'Pred']].head(10).to_string(index=False))

Submission saved: 132133 rows
  Min:  0.0015
  Max:  0.9988
  Mean: 0.4996

Sample predictions:
            ID     Pred
2026_1101_1102 0.835061
2026_1101_1103 0.081702
2026_1101_1104 0.063647
2026_1101_1105 0.702569
2026_1101_1106 0.738366
2026_1101_1107 0.662847
2026_1101_1108 0.838029
2026_1101_1110 0.482264
2026_1101_1111 0.366664
2026_1101_1112 0.012776


In [11]:
# --- ESPN Bracket Optimizer ---
# Monte Carlo simulation + recursive EV optimization for ESPN scoring
# ESPN points: 10/20/40/80/160/320 per correct pick in rounds 1-6

NUMBER_OF_SIMULATIONS = 100_000
ESPN_ROUND_POINTS = {1: 10, 2: 20, 3: 40, 4: 80, 5: 160, 6: 320}
ROUND_NAMES = {1: 'Round of 64', 2: 'Round of 32', 3: 'Sweet 16',
               4: 'Elite 8', 5: 'Final Four', 6: 'Championship'}
ROUND_GAMES = {1: 32, 2: 16, 3: 8, 4: 4, 5: 2, 6: 1}

region_labels = {'W': 'WEST', 'X': 'EAST', 'Y': 'SOUTH', 'Z': 'MIDWEST'}
gender_labels = {'M': "MEN'S", 'W': "WOMEN'S"}

for gender in GENDERS:
    start_time = time.time()

    pipeline = pipelines[gender]
    model = final_models[gender]
    seed_lookup = pipeline['seed_lookup']
    predict_matchup_fn = pipeline['predict_matchup']
    precompute_team_data = pipeline['precompute_team_data']
    team_names = dict(zip(data[f'{gender}Teams']['TeamID'],
                          data[f'{gender}Teams']['TeamName']))

    # Build seed_to_team for current season
    seed_to_team = {}
    current_seeds = data[f'{gender}NCAATourneySeeds']
    for _, row in current_seeds[current_seeds['Season'] == CURRENT_SEASON].iterrows():
        seed_to_team[row['Seed']] = row['TeamID']

    # Precompute features/ordinals for all tournament teams
    features_cache, ordinals_cache = precompute_team_data(set(seed_to_team.values()))

    # Get all tournament slots for current season
    slots_current = data[f'{gender}NCAATourneySlots']
    slots_current = slots_current[slots_current['Season'] == CURRENT_SEASON]

    # Resolve play-in games (slots that don't start with 'R')
    playin_slots = slots_current[~slots_current['Slot'].str.startswith('R')]
    playin_winners = {}
    if len(playin_slots) > 0:
        for _, slot in playin_slots.iterrows():
            strong_team = seed_to_team[slot['StrongSeed']]
            weak_team = seed_to_team[slot['WeakSeed']]
            strong_seed_num = seed_lookup[(CURRENT_SEASON, strong_team)]
            weak_seed_num = seed_lookup[(CURRENT_SEASON, weak_team)]
            prob_strong_wins = predict_matchup_fn(
                model, strong_team, weak_team, strong_seed_num, weak_seed_num,
                precomputed_features=features_cache, precomputed_ordinals=ordinals_cache)
            playin_winners[slot['Slot']] = strong_team if prob_strong_wins > 0.5 else weak_team

    # Build bracket tree and R1 matchups
    bracket_slots = slots_current[slots_current['Slot'].str.startswith('R')]
    bracket_tree = {}
    r1_matchups = {}
    all_tournament_teams = set()

    for _, slot in bracket_slots.iterrows():
        slot_name = slot['Slot']
        round_number = int(slot_name[1])

        if round_number == 1:
            strong_seed_str = slot['StrongSeed']
            weak_seed_str = slot['WeakSeed']
            strong_team = playin_winners.get(strong_seed_str, seed_to_team.get(strong_seed_str))
            weak_team = playin_winners.get(weak_seed_str, seed_to_team.get(weak_seed_str))
            r1_matchups[slot_name] = (strong_team, weak_team)
            bracket_tree[slot_name] = {
                'strong_child': None, 'weak_child': None, 'round_number': 1,
            }
            all_tournament_teams.add(strong_team)
            all_tournament_teams.add(weak_team)
        else:
            bracket_tree[slot_name] = {
                'strong_child': slot['StrongSeed'],
                'weak_child': slot['WeakSeed'],
                'round_number': round_number,
            }

    # Build team_to_r1_slot: team_id -> their R1 slot
    team_to_r1_slot = {}
    for slot_name, (strong_team, weak_team) in r1_matchups.items():
        team_to_r1_slot[strong_team] = slot_name
        team_to_r1_slot[weak_team] = slot_name

    # Build slot_descendant_r1: for each slot, the set of R1 slots that feed into it
    slot_descendant_r1 = {}

    def compute_descendant_r1(slot_name):
        if slot_name in slot_descendant_r1:
            return slot_descendant_r1[slot_name]
        node = bracket_tree[slot_name]
        if node['round_number'] == 1:
            result = {slot_name}
        else:
            result = (compute_descendant_r1(node['strong_child'])
                      | compute_descendant_r1(node['weak_child']))
        slot_descendant_r1[slot_name] = result
        return result

    for slot_name in bracket_tree:
        compute_descendant_r1(slot_name)

    def get_source_child(slot_name, team_id):
        """Determine which child subtree a team originates from."""
        node = bracket_tree[slot_name]
        team_r1 = team_to_r1_slot[team_id]
        if team_r1 in slot_descendant_r1[node['strong_child']]:
            return node['strong_child']
        return node['weak_child']

    # --- Pairwise probability matrix ---
    tournament_teams = sorted(all_tournament_teams)
    team_to_index = {team: i for i, team in enumerate(tournament_teams)}
    n_teams = len(tournament_teams)
    prob_matrix = np.full((n_teams, n_teams), 0.5)

    for i, team_a in enumerate(tournament_teams):
        seed_a = seed_lookup[(CURRENT_SEASON, team_a)]
        for j in range(i + 1, n_teams):
            team_b = tournament_teams[j]
            seed_b = seed_lookup[(CURRENT_SEASON, team_b)]
            prob_a_wins = predict_matchup_fn(
                model, team_a, team_b, seed_a, seed_b,
                precomputed_features=features_cache, precomputed_ordinals=ordinals_cache)
            prob_matrix[i, j] = prob_a_wins
            prob_matrix[j, i] = 1 - prob_a_wins

    pairwise_time = time.time()
    print(f"[{gender}] Pairwise probabilities: {n_teams} teams, "
          f"{n_teams * (n_teams - 1) // 2} matchups ({pairwise_time - start_time:.1f}s)")

    # --- Monte Carlo simulation (vectorized with numpy) ---
    # Optimization note: further speedup possible via full numpy vectorization per round
    sorted_slots = sorted(bracket_tree.keys(), key=lambda s: (int(s[1]), s))
    slot_index = {slot: i for i, slot in enumerate(sorted_slots)}
    random_draws = np.random.random((NUMBER_OF_SIMULATIONS, len(sorted_slots)))
    slot_winner_indices = {}

    for slot_name in sorted_slots:
        node = bracket_tree[slot_name]
        draws = random_draws[:, slot_index[slot_name]]

        if node['round_number'] == 1:
            strong_team, weak_team = r1_matchups[slot_name]
            strong_idx = team_to_index[strong_team]
            weak_idx = team_to_index[weak_team]
            prob_strong = prob_matrix[strong_idx, weak_idx]
            winners = np.where(draws < prob_strong, strong_idx, weak_idx)
        else:
            strong_child_winners = slot_winner_indices[node['strong_child']]
            weak_child_winners = slot_winner_indices[node['weak_child']]
            probs = prob_matrix[strong_child_winners, weak_child_winners]
            winners = np.where(draws < probs, strong_child_winners, weak_child_winners)

        slot_winner_indices[slot_name] = winners

    # Convert simulation counts to probabilities
    slot_team_probability = {}
    for slot_name in sorted_slots:
        unique_indices, counts = np.unique(slot_winner_indices[slot_name], return_counts=True)
        slot_team_probability[slot_name] = {
            tournament_teams[idx]: count / NUMBER_OF_SIMULATIONS
            for idx, count in zip(unique_indices, counts)
        }

    sim_time = time.time()
    print(f"[{gender}] Monte Carlo: {NUMBER_OF_SIMULATIONS:,} simulations ({sim_time - pairwise_time:.1f}s)")

    # --- Recursive EV optimizer ---
    unconstrained_cache = {}
    constrained_cache = {}

    def optimize_subtree(slot_name):
        """Unconstrained optimization: returns (expected_value, picks_dict)."""
        if slot_name in unconstrained_cache:
            return unconstrained_cache[slot_name]

        node = bracket_tree[slot_name]
        round_number = node['round_number']
        round_points = ESPN_ROUND_POINTS[round_number]

        if round_number == 1:
            strong_team, weak_team = r1_matchups[slot_name]
            prob_strong = slot_team_probability[slot_name].get(strong_team, 0)
            prob_weak = slot_team_probability[slot_name].get(weak_team, 0)
            if prob_strong >= prob_weak:
                result = (prob_strong * round_points, {slot_name: strong_team})
            else:
                result = (prob_weak * round_points, {slot_name: weak_team})
            unconstrained_cache[slot_name] = result
            return result

        best_ev = -1
        best_picks = None

        for team_id in slot_team_probability[slot_name]:
            this_slot_ev = slot_team_probability[slot_name][team_id] * round_points
            source_child = get_source_child(slot_name, team_id)
            other_child = (node['weak_child'] if source_child == node['strong_child']
                           else node['strong_child'])

            source_ev, source_picks = optimize_subtree_forced(source_child, team_id)
            other_ev, other_picks = optimize_subtree(other_child)
            total_ev = this_slot_ev + source_ev + other_ev

            if total_ev > best_ev:
                best_ev = total_ev
                best_picks = {slot_name: team_id}
                best_picks.update(source_picks)
                best_picks.update(other_picks)

        result = (best_ev, best_picks)
        unconstrained_cache[slot_name] = result
        return result

    def optimize_subtree_forced(slot_name, forced_team):
        """Constrained optimization: forced_team must win this slot."""
        cache_key = (slot_name, forced_team)
        if cache_key in constrained_cache:
            return constrained_cache[cache_key]

        node = bracket_tree[slot_name]
        round_number = node['round_number']
        round_points = ESPN_ROUND_POINTS[round_number]
        this_slot_ev = slot_team_probability[slot_name].get(forced_team, 0) * round_points

        if round_number == 1:
            result = (this_slot_ev, {slot_name: forced_team})
            constrained_cache[cache_key] = result
            return result

        source_child = get_source_child(slot_name, forced_team)
        other_child = (node['weak_child'] if source_child == node['strong_child']
                       else node['strong_child'])

        source_ev, source_picks = optimize_subtree_forced(source_child, forced_team)
        other_ev, other_picks = optimize_subtree(other_child)

        total_ev = this_slot_ev + source_ev + other_ev
        picks = {slot_name: forced_team}
        picks.update(source_picks)
        picks.update(other_picks)

        result = (total_ev, picks)
        constrained_cache[cache_key] = result
        return result

    # Run optimizer from championship slot
    championship_slot = [s for s in sorted_slots if s.startswith('R6')][0]
    total_expected_value, optimal_picks = optimize_subtree(championship_slot)

    opt_time = time.time()
    print(f"[{gender}] Optimizer: {len(unconstrained_cache)} unconstrained, "
          f"{len(constrained_cache)} constrained entries ({opt_time - sim_time:.1f}s)")

    # --- Display: region-by-region bracket ---
    final_four_slots = sorted([bracket_tree[championship_slot]['strong_child'],
                               bracket_tree[championship_slot]['weak_child']])

    print(f"\n{'=' * 70}")
    print(f"  {gender_labels[gender]} ESPN BRACKET OPTIMIZER")
    print(f"  ({NUMBER_OF_SIMULATIONS:,} simulations)")
    print(f"{'=' * 70}")

    for region_code in ['W', 'X', 'Y', 'Z']:
        print(f"\n--- {region_labels[region_code]} ---")
        print(f"  {'Round':<6s} {'Pick':<28s} {'Seed':>4s} {'P(win)':>8s} {'EV':>6s}")
        print(f"  {'─' * 54}")

        for round_num in range(1, 5):
            round_points = ESPN_ROUND_POINTS[round_num]
            region_slots = sorted(s for s in sorted_slots
                                  if int(s[1]) == round_num and s[2] == region_code)
            for slot_name in region_slots:
                team_id = optimal_picks[slot_name]
                name = team_names[team_id]
                seed_num = seed_lookup[(CURRENT_SEASON, team_id)]
                prob = slot_team_probability[slot_name].get(team_id, 0)
                ev = prob * round_points
                round_label = f"R{round_num}" if slot_name == region_slots[0] else ""
                print(f"  {round_label:<6s} {name:<28s} ({seed_num:>2d}) {prob:>7.1%} {ev:>6.1f}")

    # Final Four & Championship
    print(f"\n--- FINAL FOUR & CHAMPIONSHIP ---")
    print(f"  {'Slot':<6s} {'Pick':<28s} {'Seed':>4s} {'P(win)':>8s} {'EV':>6s}")
    print(f"  {'─' * 54}")
    for slot_name in final_four_slots + [championship_slot]:
        team_id = optimal_picks[slot_name]
        name = team_names[team_id]
        seed_num = seed_lookup[(CURRENT_SEASON, team_id)]
        round_num = int(slot_name[1])
        round_points = ESPN_ROUND_POINTS[round_num]
        prob = slot_team_probability[slot_name].get(team_id, 0)
        ev = prob * round_points
        print(f"  {slot_name:<6s} {name:<28s} ({seed_num:>2d}) {prob:>7.1%} {ev:>6.1f}")

    # --- Round-by-round EV breakdown ---
    print(f"\n--- EXPECTED POINTS BY ROUND ---")
    print(f"  {'Round':<16s} {'Games':>5s} {'Max Pts':>8s} {'Exp Pts':>8s} {'Pct':>6s}")
    print(f"  {'─' * 45}")

    grand_total_ev = 0
    grand_total_max = 0
    for round_num in range(1, 7):
        round_points = ESPN_ROUND_POINTS[round_num]
        games = ROUND_GAMES[round_num]
        max_points = games * round_points
        round_ev = sum(
            slot_team_probability[s].get(optimal_picks[s], 0) * round_points
            for s in sorted_slots if int(s[1]) == round_num)
        grand_total_ev += round_ev
        grand_total_max += max_points
        print(f"  {ROUND_NAMES[round_num]:<16s} {games:>5d} {max_points:>8d} "
              f"{round_ev:>8.1f} {round_ev / max_points:>5.1%}")

    print(f"  {'─' * 45}")
    print(f"  {'TOTAL':<16s} {63:>5d} {grand_total_max:>8d} "
          f"{grand_total_ev:>8.1f} {grand_total_ev / grand_total_max:>5.1%}")

    # --- Top 15 teams by total EV contribution ---
    team_ev_contribution = defaultdict(float)
    team_deepest_round = defaultdict(int)

    for slot_name in sorted_slots:
        team_id = optimal_picks[slot_name]
        round_num = int(slot_name[1])
        round_points = ESPN_ROUND_POINTS[round_num]
        prob = slot_team_probability[slot_name].get(team_id, 0)
        team_ev_contribution[team_id] += prob * round_points
        team_deepest_round[team_id] = max(team_deepest_round[team_id], round_num)

    print(f"\n--- TOP 15 TEAMS BY EV CONTRIBUTION ---")
    print(f"  {'Team':<28s} {'Seed':>4s} {'Deepest':>14s} {'Total EV':>9s}")
    print(f"  {'─' * 57}")

    for team_id, ev in sorted(team_ev_contribution.items(), key=lambda x: -x[1])[:15]:
        name = team_names[team_id]
        seed_num = seed_lookup[(CURRENT_SEASON, team_id)]
        deepest = ROUND_NAMES[team_deepest_round[team_id]]
        print(f"  {name:<28s} ({seed_num:>2d}) {deepest:>14s} {ev:>8.1f}")

    print(f"\n  Bracket slots filled: {len(optimal_picks)}")
    print(f"  Total expected value: {grand_total_ev:.1f} / {grand_total_max}")
    print(f"  Total time: {time.time() - start_time:.1f}s")

[M] Pairwise probabilities: 64 teams, 2016 matchups (4.3s)
[M] Monte Carlo: 100,000 simulations (0.1s)
[M] Optimizer: 63 unconstrained, 292 constrained entries (0.0s)

  MEN'S ESPN BRACKET OPTIMIZER
  (100,000 simulations)

--- WEST ---
  Round  Pick                         Seed   P(win)     EV
  ──────────────────────────────────────────────────────
  R1     Duke                         ( 1)   97.5%    9.7
         Connecticut                  ( 2)   93.8%    9.4
         Michigan St                  ( 3)   92.5%    9.2
         Kansas                       ( 4)   81.1%    8.1
         St John's                    ( 5)   81.5%    8.2
         Louisville                   ( 6)   60.2%    6.0
         UCLA                         ( 7)   58.9%    5.9
         Ohio St                      ( 8)   51.7%    5.2
  R2     Duke                         ( 1)   83.2%   16.6
         Connecticut                  ( 2)   67.6%   13.5
         Michigan St                  ( 3)   63.7%   12.7
         